In [1]:
%load_ext autoreload
%autoreload 2

In [12]:
import os
import multiprocessing

print(f"CPU cores: {multiprocessing.cpu_count()}")
print(f"RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3):.1f} GB")

CPU cores: 8
RAM: 16.0 GB


In [ ]:
!git clone https://github.com/emadonev/nbody_exoplanets.git

In [ ]:
%cd /content/nbody_exoplanets
!pip install -e .

In [2]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from numba import jit
import scipy
import pandas as pd

In [3]:
planets = pd.read_csv("../input/planets_triple.csv")

In [4]:
triples = pd.read_csv('../input/final_triple_5.csv')

In [5]:
from integrator import run_system
from integrator.experiments import generate_habitability_experiments

In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
output_dir = "/content/drive/MyDrive/nbody_outputs"
os.makedirs(output_dir, exist_ok=True)

Mounted at /content/drive


In [6]:
from integrator.batch import recommended_dt

configs = generate_habitability_experiments(
    triples,
    planets,
    tf=1e5,
    output_dir='../output',
)

# Compute dt and output cadence for each config
for cfg in configs:
    cfg["dt"] = recommended_dt(cfg)
    cfg["output_every_n"] = max(1, int(1.0 / cfg["dt"]))  # ~1 snapshot per year

print(f"Generated {len(configs)} experiment configs")
print(f"tf = {configs[0]['tf']:.0f} yr")
print(f"Example dt = {configs[0]['dt']:.6g} yr")
print(f"Example output_every_n = {configs[0]['output_every_n']}")
print(f"Example snapshots = {int(configs[0]['tf'] / configs[0]['dt'] / configs[0]['output_every_n'])}")

Generated 101 experiment configs
tf = 100000 yr
Example dt = 0.00203401 yr
Example output_every_n = 491
Example snapshots = 100130


In [7]:
preview_df = pd.DataFrame([
    {
        'system_name': cfg['system_name'],
        'planet_scenario': cfg['planet_scenario'],
        'host_star': cfg['host_star'],
        'system_type': cfg['system_type'],
        'outer_e': cfg['outer_e'],
        'outer_i': cfg['outer_i'],
        'source_outer_e': cfg['source_outer_e'],
        'source_outer_i': cfg['source_outer_i'],
        'source_inner_e': cfg['source_inner_e'],
        'source_inner_i': cfg['source_inner_i'],
        'n_planets': len(cfg['planets']),
        'experiment_name': cfg['experiment_name'],
        'output_file': cfg['output_file'],
    }
    for cfg in configs
])

summary_df = (
    preview_df.groupby(['system_name', 'planet_scenario'], as_index=False)
    .agg(
        n_runs=('experiment_name', 'size'),
        outer_e_values=('outer_e', lambda s: sorted(pd.unique(s))),
        outer_i_values=('outer_i', lambda s: sorted(pd.unique(s))),
        n_planets=('n_planets', 'first'),
    )
    .sort_values(['system_name', 'planet_scenario'])
)

print(f"Total configs: {len(preview_df)}")
display(summary_df)
display(preview_df.head(20))

Total configs: 101


,system_name,planet_scenario,n_runs,outer_e_values,outer_i_values,n_planets
0,94 Ceti,hz_inner,1,[0.26],[104.0],2
1,94 Ceti,hz_mid,1,[0.26],[104.0],2
2,94 Ceti,hz_outer,1,[0.26],[104.0],2
3,Gliese 667,observed,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",5
4,HD 132563,hz_inner,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",2
5,HD 132563,hz_mid,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",2
6,HD 132563,hz_outer,20,"[0.0, 0.2, 0.4, 0.6, 0.8]","[0.0, 30.0, 60.0, 90.0]",2
7,Kepler-444,hz_inner,1,[0.55],[85.4],6
8,Kepler-444,hz_mid,1,[0.55],[85.4],6
9,Kepler-444,hz_outer,1,[0.55],[85.4],6


,system_name,planet_scenario,host_star,system_type,outer_e,outer_i,source_outer_e,source_outer_i,source_inner_e,source_inner_i,n_planets,experiment_name,output_file
0,Gliese 667,observed,C,S(C),0.0,0.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv000,../output/Gliese_667_observed_ev0.0_iv000.hdf5
1,Gliese 667,observed,C,S(C),0.0,30.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv030,../output/Gliese_667_observed_ev0.0_iv030.hdf5
2,Gliese 667,observed,C,S(C),0.0,60.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv060,../output/Gliese_667_observed_ev0.0_iv060.hdf5
3,Gliese 667,observed,C,S(C),0.0,90.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.0_iv090,../output/Gliese_667_observed_ev0.0_iv090.hdf5
4,Gliese 667,observed,C,S(C),0.2,0.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv000,../output/Gliese_667_observed_ev0.2_iv000.hdf5
5,Gliese 667,observed,C,S(C),0.2,30.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv030,../output/Gliese_667_observed_ev0.2_iv030.hdf5
6,Gliese 667,observed,C,S(C),0.2,60.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv060,../output/Gliese_667_observed_ev0.2_iv060.hdf5
7,Gliese 667,observed,C,S(C),0.2,90.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.2_iv090,../output/Gliese_667_observed_ev0.2_iv090.hdf5
8,Gliese 667,observed,C,S(C),0.4,0.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.4_iv000,../output/Gliese_667_observed_ev0.4_iv000.hdf5
9,Gliese 667,observed,C,S(C),0.4,30.0,NaN,NaN,0.57,127.6,5,Gliese_667_observed_ev0.4_iv030,../output/Gliese_667_observed_ev0.4_iv030.hdf5


In [10]:
import os

In [11]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp

max_workers = min(2, mp.cpu_count())
batch_size = max_workers * 4
resume_existing = True
test_mode = False
test_limit = 1

if resume_existing:
    jobs = [cfg for cfg in configs if not os.path.exists(cfg['output_file'])]
else:
    jobs = list(configs)

if test_mode:
    jobs = jobs[:test_limit]

print(f"Workers: {max_workers}")
print(f"Batch size: {batch_size}")
print(f"Jobs queued: {len(jobs)}")

Workers: 2
Batch size: 8
Jobs queued: 15


In [12]:
results = []
errors = []

with ProcessPoolExecutor(
    max_workers=max_workers,
    mp_context=mp.get_context('spawn'),
) as pool:
    future_to_cfg = {
        pool.submit(run_system, cfg): cfg
        for cfg in jobs
    }

    for future in as_completed(future_to_cfg):
        cfg = future_to_cfg[future]
        name = cfg['experiment_name']
        try:
            output_path = future.result()
            results.append({
                'experiment_name': name,
                'system_name': cfg['system_name'],
                'planet_scenario': cfg['planet_scenario'],
                'output_file': output_path,
            })
            print(f"OK: {name}")
        except Exception as exc:
            errors.append({
                'experiment_name': name,
                'system_name': cfg['system_name'],
                'planet_scenario': cfg['planet_scenario'],
                'error': repr(exc),
            })
            print(f"FAIL: {name} -> {exc}")

results_df = pd.DataFrame(results)
errors_df = pd.DataFrame(errors)

print(f"\nCompleted: {len(results_df)}")
print(f"Errors: {len(errors_df)}")
if len(errors_df) > 0:
    display(errors_df)

OK: LTT_1445_hz_inner_ev0.0_iv092
OK: LTT_1445_hz_inner_ev0.2_iv092
FAIL: LTT_1445_hz_inner_ev0.6_iv092 -> Kepler universal solve did not converge
FAIL: LTT_1445_hz_inner_ev0.4_iv092 -> Kepler universal solve did not converge
FAIL: LTT_1445_hz_mid_ev0.0_iv092 -> Kepler universal solve did not converge
OK: LTT_1445_hz_inner_ev0.8_iv092
OK: LTT_1445_hz_mid_ev0.2_iv092
FAIL: LTT_1445_hz_mid_ev0.4_iv092 -> Kepler universal solve did not converge
FAIL: LTT_1445_hz_mid_ev0.8_iv092 -> Kepler universal solve did not converge
FAIL: LTT_1445_hz_outer_ev0.0_iv092 -> Kepler universal solve did not converge
OK: LTT_1445_hz_mid_ev0.6_iv092
OK: LTT_1445_hz_outer_ev0.2_iv092
OK: LTT_1445_hz_outer_ev0.4_iv092
OK: LTT_1445_hz_outer_ev0.6_iv092
FAIL: LTT_1445_hz_outer_ev0.8_iv092 -> Kepler universal solve did not converge

Completed: 8
Errors: 7


,experiment_name,system_name,planet_scenario,error
0,LTT_1445_hz_inner_ev0.6_iv092,LTT 1445,hz_inner,RuntimeError('Kepler universal solve did not c...
1,LTT_1445_hz_inner_ev0.4_iv092,LTT 1445,hz_inner,RuntimeError('Kepler universal solve did not c...
2,LTT_1445_hz_mid_ev0.0_iv092,LTT 1445,hz_mid,RuntimeError('Kepler universal solve did not c...
3,LTT_1445_hz_mid_ev0.4_iv092,LTT 1445,hz_mid,RuntimeError('Kepler universal solve did not c...
4,LTT_1445_hz_mid_ev0.8_iv092,LTT 1445,hz_mid,RuntimeError('Kepler universal solve did not c...
5,LTT_1445_hz_outer_ev0.0_iv092,LTT 1445,hz_outer,RuntimeError('Kepler universal solve did not c...
6,LTT_1445_hz_outer_ev0.8_iv092,LTT 1445,hz_outer,RuntimeError('Kepler universal solve did not c...


In [ ]:
# --- Post-processing: compute orbital elements, energy, temperature, HZ ---
from integrator import postprocess

for i, cfg in enumerate(configs):
    path = cfg['output_file']
    if os.path.exists(path):
        print(f"[{i+1}/{len(configs)}] Post-processing: {cfg['experiment_name']}")
        try:
            postprocess(path)
        except Exception as exc:
            print(f"  FAIL: {exc}")
    else:
        print(f"[{i+1}/{len(configs)}] Missing: {cfg['experiment_name']}")

print("Post-processing complete.")